In [8]:
# ============================================================
# FINAL HS4 CODE CLASSIFIER
# Word + Character TF-IDF + Calibrated Linear SVM
# ============================================================

import ast
import os
import re
import joblib
import numpy as np
import pandas as pd

from datasets import load_dataset

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import accuracy_score, classification_report


# ============================================================
# 1. LOAD DATA
# ============================================================

print("Loading dataset...")

df = pd.DataFrame(
    load_dataset(
        "Dayanand314Krishna/cross_rulings_hts_dataset_for_tariffs"
    )["train"]
)

print("Original shape:", df.shape)


# ============================================================
# 2. EXTRACT USER TEXT + HS4 CODE
# ============================================================

def extract_text_and_code(message):

    try:
        data = ast.literal_eval(str(message))
    except:
        data = []

    user_text = ""
    assistant_text = ""

    if isinstance(data, list):
        for m in data:
            if m.get("role") == "user" and not user_text:
                user_text = m.get("content", "")

            elif m.get("role") == "assistant" and not assistant_text:
                assistant_text = m.get("content", "")

    # Remove question wording
    user_text = re.sub(
        r"^(what\s+is\s+the\s+hts\s+(us\s+)?code\s+for\s+|"
        r"what\s+is\s+the\s+hs\s+code\s+for\s+)",
        "",
        user_text,
        flags=re.IGNORECASE
    ).rstrip("?")

    # Extract HS4 from assistant answer
    match = re.search(
        r"\b(\d{4})\.\d{2}",
        assistant_text
    )

    if not match:
        match = re.search(
            r"\b(\d{4})\b",
            assistant_text
        )

    hs4 = match.group(1) if match else None

    return user_text.lower().strip(), hs4


parsed = df["messages"].apply(
    extract_text_and_code
)

df["clean_text"] = parsed.apply(
    lambda x: x[0]
)

df["hts_code"] = parsed.apply(
    lambda x: x[1]
)


# ============================================================
# 3. CLEAN DATA
# ============================================================

df = df.dropna(
    subset=["clean_text", "hts_code"]
).copy()

# Extract first 4 digits
df["hs4"] = (
    df["hts_code"]
    .astype(str)
    .str.replace(
        r"[^\d]",
        "",
        regex=True
    )
    .str[:4]
)

# Basic text cleaning
df["clean_text"] = (
    df["clean_text"]
    .astype(str)
    .str.lower()
    .str.strip()
)

# Keep valid HS4 and meaningful text
df = df[
    df["hs4"].str.fullmatch(r"\d{4}") &
    (df["clean_text"].str.len() > 10)
].copy()


# ------------------------------------------------------------
# Remove identical text with different HS4 labels
# ------------------------------------------------------------

conflicts = (
    df.groupby("clean_text")["hs4"]
    .nunique()
)

ambiguous_text = conflicts[
    conflicts > 1
].index

df = df[
    ~df["clean_text"].isin(ambiguous_text)
].copy()


# ------------------------------------------------------------
# Remove exact duplicate examples
# ------------------------------------------------------------

df = df.drop_duplicates(
    subset=["clean_text", "hs4"]
).reset_index(drop=True)


# ------------------------------------------------------------
# Remove very rare HS4 classes
# ------------------------------------------------------------

MIN_SAMPLES = 10

valid_classes = (
    df["hs4"]
    .value_counts()
    .loc[
        lambda x: x >= MIN_SAMPLES
    ]
    .index
)

df = df[
    df["hs4"].isin(valid_classes)
].copy()

print("\nFinal dataset shape:", df.shape)
print("HS4 classes:", df["hs4"].nunique())


# ============================================================
# 4. TRAIN / TEST SPLIT
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    df["clean_text"],
    df["hs4"],
    test_size=0.20,
    random_state=42,
    stratify=df["hs4"]
)

print("Training samples:", len(X_train))
print("Testing samples :", len(X_test))


# ============================================================
# 5. FINAL MODEL
# ============================================================

model = Pipeline([

    (
        "features",

        FeatureUnion([

            # Word-level TF-IDF
            (
                "word",

                TfidfVectorizer(
                    analyzer="word",
                    ngram_range=(1, 2),
                    min_df=2,
                    max_features=30000,
                    sublinear_tf=True
                )
            ),

            # Character-level TF-IDF
            (
                "char",

                TfidfVectorizer(
                    analyzer="char_wb",
                    ngram_range=(3, 5),
                    min_df=2,
                    max_features=30000,
                    sublinear_tf=True
                )
            )
        ])
    ),

    (
        "svm",

        CalibratedClassifierCV(
            LinearSVC(
                C=1.0,
                class_weight="balanced",
                max_iter=5000,
                random_state=42
            ),
            cv=3,
            method="sigmoid",
            n_jobs=-1
        )
    )
])


# ============================================================
# 6. TRAIN MODEL
# ============================================================

print("\nTraining final calibrated SVM...")

model.fit(
    X_train,
    y_train
)


# ============================================================
# 7. EVALUATION
# ============================================================

probabilities = model.predict_proba(
    X_test
)

classes = model.classes_


# ------------------------------------------------------------
# Top-1 prediction
# ------------------------------------------------------------

top1_index = np.argmax(
    probabilities,
    axis=1
)

predictions = classes[
    top1_index
]


# ------------------------------------------------------------
# Top-3 predictions
# ------------------------------------------------------------

top3_index = np.argsort(
    probabilities,
    axis=1
)[:, -3:][:, ::-1]

top3_labels = classes[
    top3_index
]


# ------------------------------------------------------------
# Accuracy
# ------------------------------------------------------------

top1_accuracy = accuracy_score(
    y_test,
    predictions
)

top3_accuracy = np.mean([
    true_label in predicted_labels

    for true_label, predicted_labels
    in zip(
        y_test,
        top3_labels
    )
])


# ------------------------------------------------------------
# Wrong predictions
# ------------------------------------------------------------

wrong_count = np.sum(
    predictions != y_test.values
)


print("\n==============================")
print("FINAL MODEL RESULTS")
print("==============================")

print(
    f"Top-1 Accuracy: "
    f"{top1_accuracy * 100:.2f}%"
)

print(
    f"Top-3 Accuracy: "
    f"{top3_accuracy * 100:.2f}%"
)

print(
    f"\nTotal test cases: "
    f"{len(y_test)}"
)

print(
    f"Correct Top-1: "
    f"{len(y_test) - wrong_count}"
)

print(
    f"Wrong Top-1: "
    f"{wrong_count}"
)


# ============================================================
# 8. CLASSIFICATION REPORT
# ============================================================

print("\n==============================")
print("CLASSIFICATION REPORT")
print("==============================")

print(
    classification_report(
        y_test,
        predictions,
        zero_division=0
    )
)


# ============================================================
# 9. SAVE TEST RESULTS
# ============================================================

results_df = pd.DataFrame({

    "text":
        X_test.values,

    "actual_hs4":
        y_test.values,

    "predicted_hs4":
        predictions,

    "top1_correct":
        predictions == y_test.values,

    "top3_predictions": [
        " | ".join(row)
        for row in top3_labels
    ],

    "confidence": [
        round(
            probabilities[
                i,
                top1_index[i]
            ] * 100,
            2
        )

        for i in range(
            len(X_test)
        )
    ]
})


# All test predictions
results_df.to_csv(
    "test_predictions.csv",
    index=False
)


# Only wrong Top-1 predictions
results_df[
    ~results_df["top1_correct"]
].to_csv(
    "wrong_predictions.csv",
    index=False
)


# ============================================================
# 10. LIVE PREDICTION
# ============================================================

sample = (
    "Women's 100% woven cotton summer dress "
    "with floral embroidery"
)

prob = model.predict_proba(
    [sample]
)[0]

top3_index = np.argsort(
    prob
)[-3:][::-1]


print("\n==============================")
print("LIVE HS4 PREDICTION")
print("==============================")

print(
    "\nProduct:",
    sample
)

print("\nTop-3 Predictions:")

for rank, idx in enumerate(
    top3_index,
    start=1
):

    print(
        f"{rank}. "
        f"HS4: {classes[idx]} "
        f"({prob[idx] * 100:.2f}%)"
    )


# ============================================================
# 11. SAVE FINAL MODEL FILES
# ============================================================

print("\n==============================")
print("SAVING MODEL FILES")
print("==============================")


# Save trained model
joblib.dump(
    model,
    "final_hs4_calibrated_svm.pkl"
)


# Save model information
model_info = {

    "model_type":
        "Word + Character TF-IDF + Calibrated Linear SVM",

    "target":
        "HS4",

    "num_classes":
        len(classes),

    "classes":
        classes.tolist(),

    "word_ngram_range":
        (1, 2),

    "character_ngram_range":
        (3, 5),

    "min_df":
        2,

    "max_word_features":
        30000,

    "max_character_features":
        30000,

    "svm_C":
        1.0,

    "class_weight":
        "balanced",

    "calibration":
        "sigmoid",

    "calibration_cv":
        3,

    "top1_accuracy":
        round(top1_accuracy, 4),

    "top3_accuracy":
        round(top3_accuracy, 4),

    "test_samples":
        len(y_test)
}


joblib.dump(
    model_info,
    "model_info.pkl"
)


# ============================================================
# 12. CHECK SAVED FILES
# ============================================================

print("\nModel files:")

for file in [
    "final_hs4_calibrated_svm.pkl",
    "model_info.pkl"
]:

    size_mb = (
        os.path.getsize(file)
        / (1024 * 1024)
    )

    print(
        f"{file}: "
        f"{size_mb:.2f} MB"
    )


print("\nResult files:")

for file in [
    "test_predictions.csv",
    "wrong_predictions.csv"
]:

    size_mb = (
        os.path.getsize(file)
        / (1024 * 1024)
    )

    print(
        f"{file}: "
        f"{size_mb:.2f} MB"
    )


print("\n==============================")
print("PROJECT COMPLETE")
print("==============================")

Loading dataset...
Original shape: (18254, 1)

Final dataset shape: (14880, 4)
HS4 classes: 252
Training samples: 11904
Testing samples : 2976

Training final calibrated SVM...

FINAL MODEL RESULTS
Top-1 Accuracy: 69.62%
Top-3 Accuracy: 81.69%

Total test cases: 2976
Correct Top-1: 2072
Wrong Top-1: 904

CLASSIFICATION REPORT
              precision    recall  f1-score   support

        2703       1.00      1.00      1.00         3
        2825       0.67      0.67      0.67         3
        2901       1.00      0.40      0.57         5
        2902       0.50      0.50      0.50         4
        2903       0.62      0.71      0.67         7
        2904       1.00      0.50      0.67         8
        2905       0.71      0.77      0.74        13
        2906       0.56      0.62      0.59         8
        2907       0.80      0.57      0.67         7
        2909       0.71      0.42      0.53        12
        2911       1.00      0.67      0.80         3
        2912       0.50